In [1]:
%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore')

#Load environment variables 
from helper import load_env
load_env()

import os
import yaml
from crewai import Agent, Task, Crew


### Set OpenAI model

In [2]:
os.environ['OPENAI_MODEL_NAME'] = "gpt-4o-mini"

### Load the task and agent ymal file

In [3]:
#Define the file path for yaml configuration
files = {
    'agents': 'config/agents.yaml',
    'tasks': 'config/tasks.yaml'
}

#Load configuration from yaml file
configs = {}

for config_type, file_path in files.items():
    with open(file_path, 'r') as file:
        configs[config_type] = yaml.safe_load(file)

#Assign loaded configuratio to specific variables
agents_config = configs['agents']
tasks_config = configs['tasks']

### Create Pydantic models for structured output

In [4]:
from typing import List
from pydantic import BaseModel, Field

class TaskEstimate(BaseModel):
    task_name: str = Field(..., description="Name of the task")
    estimated_time_hours: float = Field(..., description="Estimated time to complete the task in hours")
    required_resources: List[str] = Field(..., description="List of resources required to complete the task")

class Milestone(BaseModel):
    milestone_name: str = Field(..., description="Name of the milestone")
    tasks: List[str] = Field(..., description="List of the task IDs associated with this milestone")

class ProjectPlan(BaseModel):
    tasks: List[TaskEstimate] = Field(..., description="List of task with their estimates")
    milestones: List[Milestone] = Field(..., description="List of project milestones") 


### Create Crew, Agents and Tasks

In [8]:
#Creating Agents
project_planning_agent = Agent(
    config=agents_config['project_planning_agent']
)

estimation_agent = Agent(
    config = agents_config['estimation_agent']
)

resource_allocation_agent = Agent(
    config = agents_config['resource_allocation_agent']
)

#Creating task
task_breakdown = Task(
    config=tasks_config['task_breakdown'],
    agent = project_planning_agent
)

time_resource_estimation = Task(
    config= tasks_config['time_resource_estimation'],
    agent=estimation_agent
)

resource_allocation = Task(
    config=tasks_config['resource_allocation'],
    agent=resource_allocation_agent,
    output_pydantic=ProjectPlan # This is the structured output we want
)

#Creating crew
crew = Crew(
    agents=[
        project_planning_agent,
        estimation_agent, 
        resource_allocation_agent
    ],
    tasks=[
        task_breakdown,
        time_resource_estimation,
        resource_allocation
    ],
    verbose=True
)

In [9]:
print(crew.tasks)

[Task(description=Carefully analyze the project_requirements for the {project_type} project and break them down into individual tasks. Define each task's scope in detail, set achievable timelines, and ensure that all dependencies are accounted for:
{project_requirements}

Team members:
{team_members}
, expected_output=A comprehensive list of tasks with detailed descriptions, timelines, dependencies, and deliverables. Your final output MUST include a Gantt chart or similar timeline visualization specific to the {project_type} project.
), Task(description=Thoroughly evaluate each task in the {project_type} project to estimate the time, resources, and effort required. Use historical data, task complexity, and available resources to provide a realistic estimation for each task.
, expected_output=A detailed estimation report outlining the time, resources, and effort required for each task in the {project_type} project. Your final report MUST include a summary of any risks or uncertainties a

### Crew's Inputs 


In [10]:
from IPython.display import display, Markdown

project = "Website"
industry = 'Technology'
project_objectives = 'Create a webiste for a small business'
team_members = """
- John Doe (Project Manager)
- Jane Doe (Software Engineer)
- Bob Smith (Designer)
- Alice Johnson (QA Engineer)
- Tom Brown (QA Engineer)
"""
project_requirements = """
- Create a responsive design that works well on desktop and mobile devices
- Implement a modern, visually appealing user interface with a clean look
- Develop a user-friendly navigation system with intuitive menu structure
- Include an "About Us" page highlighting the company's history and values
- Design a "Services" page showcasing the business's offerings with descriptions
- Create a "Contact Us" page with a form and integrated map for communication
- Implement a blog section for sharing industry news and company updates
- Ensure fast loading times and optimize for search engines (SEO)
- Integrate social media links and sharing capabilities
- Include a testimonials section to showcase customer feedback and build trust
"""

#formate the disctionary as markdown for a better display in jupyter lab
formatted_output = f"""
**Project Type:** {project}

**Project Objectives:** {project_objectives}

**Industry:** {industry}

**Team Members:**
{team_members}

**Project Requirements:**
{project_requirements}

"""

#Display the formatted output as markdown
display(Markdown(formatted_output))


**Project Type:** Website

**Project Objectives:** Create a webiste for a small business

**Industry:** Technology

**Team Members:**

- John Doe (Project Manager)
- Jane Doe (Software Engineer)
- Bob Smith (Designer)
- Alice Johnson (QA Engineer)
- Tom Brown (QA Engineer)


**Project Requirements:**

- Create a responsive design that works well on desktop and mobile devices
- Implement a modern, visually appealing user interface with a clean look
- Develop a user-friendly navigation system with intuitive menu structure
- Include an "About Us" page highlighting the company's history and values
- Design a "Services" page showcasing the business's offerings with descriptions
- Create a "Contact Us" page with a form and integrated map for communication
- Implement a blog section for sharing industry news and company updates
- Ensure fast loading times and optimize for search engines (SEO)
- Integrate social media links and sharing capabilities
- Include a testimonials section to showcase customer feedback and build trust




### Kicking off the crew

In [11]:
#The given pythong dictionary
inputs = {
    'project_type': project,
    'project_objectives': project_objectives,
    'industry': industry,
    'team_members': team_members,
    'project_requirements': project_requirements
}

#Run the crew
result = crew.kickoff(
    inputs = inputs
)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  55b0ea1b-a5ae-4695-9ce3-2c278e0565f0                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Carefully analyze the project_requirements for the Website project and break them down into individual   │
│  tasks. Define each task's scope in detail, set achievable timelines, and ensure that all dependencies are      │
│  accounted for:                                                                                                 │
│                                                                                                                 │
│  - Create a responsive design that works well on desktop and mobile devices                                     │
│  - Implement a modern, visually appealing user interface with a clean look                                      │
│  - Develop a user-friendly navigation system with intuitive menu structure                                      │
│  - Include an "About Us" page highlighting the company's history and values                                     │
│  - Design a "Services" page showcasing the business's offerings with descriptions                               │
│  - Create a "Contact Us" page with a form and integrated map for communication                                  │
│  - Implement a blog section for sharing industry news and company updates                                       │
│  - Ensure fast loading times and optimize for search engines (SEO)                                              │
│  - Integrate social media links and sharing capabilities                                                        │
│  - Include a testimonials section to showcase customer feedback and build trust                                 │
│                                                                                                                 │
│                                                                                                                 │
│  Team members:                                                                                                  │
│                                                                                                                 │
│  - John Doe (Project Manager)                                                                                   │
│  - Jane Doe (Software Engineer)                                                                                 │
│  - Bob Smith (Designer)                                                                                         │
│  - Alice Johnson (QA Engineer)                                                                                  │
│  - Tom Brown (QA Engineer)                                                                                      │
│                                                                                                                 │
│                                                                                                                 │
│  ID: 0a656066-e972-42b3-8d0f-df7edb0a2338                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: The Ultimate Project Planner                                                                            │
│                                                                                                                 │
│  Task: Carefully analyze the project_requirements for the Website project and break them down into individual   │
│  tasks. Define each task's scope in detail, set achievable timelines, and ensure that all dependencies are      │
│  accounted for:                                                                                                 │
│                                                                                                                 │
│  - Create a responsive design that works well on desktop and mobile devices                                     │
│  - Implement a modern, visually appealing user interface with a clean look                                      │
│  - Develop a user-friendly navigation system with intuitive menu structure                                      │
│  - Include an "About Us" page highlighting the company's history and values                                     │
│  - Design a "Services" page showcasing the business's offerings with descriptions                               │
│  - Create a "Contact Us" page with a form and integrated map for communication                                  │
│  - Implement a blog section for sharing industry news and company updates                                       │
│  - Ensure fast loading times and optimize for search engines (SEO)                                              │
│  - Integrate social media links and sharing capabilities                                                        │
│  - Include a testimonials section to showcase customer feedback and build trust                                 │
│                                                                                                                 │
│                                                                                                                 │
│  Team members:                                                                                                  │
│                                                                                                                 │
│  - John Doe (Project Manager)                                                                                   │
│  - Jane Doe (Software Engineer)                                                                                 │
│  - Bob Smith (Designer)                                                                                         │
│  - Alice Johnson (QA Engineer)                                                                                  │
│  - Tom Brown (QA Engineer)                                                                                      │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: The Ultimate Project Planner                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Website Project Breakdown and Task Plan**                                                                    │
│                                                                                                                 │
│  **Project Overview:**                                                                                          │
│  The goal is to create a fully functional, visually appealing website for a small business, ensuring            │
│  responsiveness across devices, an engaging user interface, and providing a seamless user experience. The       │
│  website will include informative pages, a blog, and integration of social media.                               │
│                                                                                                                 │
│  **Team Members:**                                                                                              │
│  - **John Doe (Project Manager)**                                                                               │
│  - **Jane Doe (Software Engineer)**                                                                             │
│  - **Bob Smith (Designer)**                                                                                     │
│  - **Alice Johnson (QA Engineer)**                                                                              │
│  - **Tom Brown (QA Engineer)**                                                                                  │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  **Task Breakdown:**                                                                                            │
│                                                                                                                 │
│  | Task ID | Task Description                                                | Scope Details | Assigned To      │
│  | Duration (days) | Start Date | End Date | Dependencies           | Deliverables                |             │
│  |---------|---------------------------------------------------------------|----------------|-----------------  │
│  ---|-----------------|------------|----------|------------------------|-----------------------------|          │
│  | 1       | Project Kickoff Meeting                                       | Align the team on objectives,      │
│  roles and timeline. | John Doe         | 1               | Day 1      | Day 1    | None                   |    │
│  Meeting notes               |                                                                                  │
│  | 2       | Requirement Gathering                                          | Collect detailed requirements     │
│  from stakeholders regarding look and functionality. | John Doe         | 3               | Day 2      | Day 4  │
│  | Task 1                 | Requirements document       |                                                       │
│  | 3       | Create wireframes and initial designs                         | Develop wireframes for all major   │
│  pages including mobile layouts. | Bob Smith        | 5

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Carefully analyze the project_requirements for the Website project and break them down into individual tasks.  │
│  Define each task's scope in detail, set achievable timelines, and ensure that all dependencies are accounted   │
│  for:                                                                                                           │
│                                                                                                                 │
│  - Create a responsive design that works well on desktop and mobile devices                                     │
│  - Implement a modern, visually appealing user interface with a clean look                                      │
│  - Develop a user-friendly navigation system with intuitive menu structure                                      │
│  - Include an "About Us" page highlighting the company's history and values                                     │
│  - Design a "Services" page showcasing the business's offerings with descriptions                               │
│  - Create a "Contact Us" page with a form and integrated map for communication                                  │
│  - Implement a blog section for sharing industry news and company updates                                       │
│  - Ensure fast loading times and optimize for search engines (SEO)                                              │
│  - Integrate social media links and sharing capabilities                                                        │
│  - Include a testimonials section to showcase customer feedback and build trust                                 │
│                                                                                                                 │
│                                                                                                                 │
│  Team members:                                                                                                  │
│                                                                                                                 │
│  - John Doe (Project Manager)                                                                                   │
│  - Jane Doe (Software Engineer)                                                                                 │
│  - Bob Smith (Designer)                                                                                         │
│  - Alice Johnson (QA Engineer)                                                                                  │
│  - Tom Brown (QA Engineer)                                                                                      │
│                                                                                                                 │
│                                                                                                                 │
│  Agent:                                                                                                         │
│  The Ultimate Project Planner                                                                                   │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰────────────────────────────────────────────────────────

...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Resource Allocation Strategist                                                                          │
│                                                                                                                 │
│  Task: Strategically allocate tasks for the Website project to team members based on their skills,              │
│  availability, and current workload. Ensure that each task is assigned to the most suitable team member and     │
│  that the workload is evenly distributed.                                                                       │
│                                                                                                                 │
│  Team members:                                                                                                  │
│                                                                                                                 │
│  - John Doe (Project Manager)                                                                                   │
│  - Jane Doe (Software Engineer)                                                                                 │
│  - Bob Smith (Designer)                                                                                         │
│  - Alice Johnson (QA Engineer)                                                                                  │
│  - Tom Brown (QA Engineer)                                                                                      │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Resource Allocation Strategist                                                                          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {                                                                                                              │
│    "tasks": [                                                                                                   │
│      {                                                                                                          │
│        "task_name": "Project Kickoff Meeting",                                                                  │
│        "estimated_time_hours": 1,                                                                               │
│        "required_resources": ["John Doe"]                                                                       │
│      },                                                                                                         │
│      {                                                                                                          │
│        "task_name": "Requirement Gathering",                                                                    │
│        "estimated_time_hours": 24,                                                                              │
│        "required_resources": ["John Doe"]                                                                       │
│      },                                                                                                         │
│      {                                                                                                          │
│        "task_name": "Create wireframes and initial designs",                                                    │
│        "estimated_time_hours": 40,                                                                              │
│        "required_resources": ["Bob Smith"]                                                                      │
│      },                                                                                                         │
│      {                                                                                                          │
│        "task_name": "Review and Feedback on Wireframes",                                                        │
│        "estimated_time_hours": 24,                                                                              │
│        "required_resources": ["John Doe"]                                                                       │
│      },                                                                                                         │
│      {                                                                                                          │
│        "task_name": "Finalize Design",                                                                          │
│        "estimated_time_hours": 16,                                                                              │
│        "required_resources": ["Bob Smith"]                                                                      │
│      },                                                                                                         │
│      {                                                                                                          │
│        "task_name": "Development of Responsive Interface",                                                      │
│        "estimated_time_hours": 80,                     

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Strategically allocate tasks for the Website project to team members based on their skills, availability, and  │
│  current workload. Ensure that each task is assigned to the most suitable team member and that the workload is  │
│  evenly distributed.                                                                                            │
│                                                                                                                 │
│  Team members:                                                                                                  │
│                                                                                                                 │
│  - John Doe (Project Manager)                                                                                   │
│  - Jane Doe (Software Engineer)                                                                                 │
│  - Bob Smith (Designer)                                                                                         │
│  - Alice Johnson (QA Engineer)                                                                                  │
│  - Tom Brown (QA Engineer)                                                                                      │
│                                                                                                                 │
│                                                                                                                 │
│  Agent:                                                                                                         │
│  Resource Allocation Strategist                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name:                                                                                                          │
│  crew                                                                                                           │
│  ID:                                                                                                            │
│  55b0ea1b-a5ae-4695-9ce3-2c278e0565f0                                                                           │
│  Final Output: {                                                                                                │
│    "tasks": [                                                                                                   │
│      {                                                                                                          │
│        "task_name": "Project Kickoff Meeting",                                                                  │
│        "estimated_time_hours": 1,                                                                               │
│        "required_resources": ["John Doe"]                                                                       │
│      },                                                                                                         │
│      {                                                                                                          │
│        "task_name": "Requirement Gathering",                                                                    │
│        "estimated_time_hours": 24,                                                                              │
│        "required_resources": ["John Doe"]                                                                       │
│      },                                                                                                         │
│      {                                                                                                          │
│        "task_name": "Create wireframes and initial designs",                                                    │
│        "estimated_time_hours": 40,                                                                              │
│        "required_resources": ["Bob Smith"]                                                                      │
│      },                                                                                                         │
│      {                                                                                                          │
│        "task_name": "Review and Feedback on Wireframes",                                                        │
│        "estimated_time_hours": 24,                                                                              │
│        "required_resources": ["John Doe"]                                                                       │
│      },                                                                                                         │
│      {                                                                                                          │
│        "task_name": "Finalize Design",                                                                          │
│        "estimated_time_hours": 16,                                                                              │
│        "required_resources": ["Bob Smith"]                                                                      │
│      },                                                                                                         │
│      {                                                

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

### Usage Metrics and costs

In [15]:
#let's see how much it would cost each time if this crew runs at scale

import pandas as pd

costs = 0.150 * (crew.usage_metrics.prompt_tokens + crew.usage_metrics.completion_tokens) /1_000_000
print(f"Total costs: ${costs: .4f}")

#Convert UsageMetics instance to a DataFrame
df_usage_metric = pd.DataFrame([crew.usage_metrics.dict()])
df_usage_metric


Total costs: $ 0.0017


,total_tokens,prompt_tokens,cached_prompt_tokens,completion_tokens,successful_requests
0,11018,6868,0,4150,3


### Result

In [14]:
result.pydantic.dict()

{'tasks': [{'task_name': 'Project Kickoff Meeting',
   'estimated_time_hours': 1.0,
   'required_resources': ['John Doe']},
  {'task_name': 'Requirement Gathering',
   'estimated_time_hours': 24.0,
   'required_resources': ['John Doe']},
  {'task_name': 'Create wireframes and initial designs',
   'estimated_time_hours': 40.0,
   'required_resources': ['Bob Smith']},
  {'task_name': 'Review and Feedback on Wireframes',
   'estimated_time_hours': 24.0,
   'required_resources': ['John Doe']},
  {'task_name': 'Finalize Design',
   'estimated_time_hours': 16.0,
   'required_resources': ['Bob Smith']},
  {'task_name': 'Development of Responsive Interface',
   'estimated_time_hours': 80.0,
   'required_resources': ['Jane Doe']},
  {'task_name': 'Develop User-Friendly Navigation',
   'estimated_time_hours': 32.0,
   'required_resources': ['Jane Doe']},
  {'task_name': "Create 'About Us' Page",
   'estimated_time_hours': 24.0,
   'required_resources': ['Jane Doe', 'Bob Smith']},
  {'task_name':

In [16]:
task = result.pydantic.dict()['tasks']
df_tasks = pd.DataFrame(task)

#Display the dataframe as an HTML table
df_tasks.style.set_table_attributes('border="1"').set_caption("Task Details").set_table_styles(
    [{'selector': 'th, td', 'props': [{'font-size', '120%'}] }]
)

,task_name,estimated_time_hours,required_resources
0,Project Kickoff Meeting,1.000000,['John Doe']
1,Requirement Gathering,24.000000,['John Doe']
2,Create wireframes and initial designs,40.000000,['Bob Smith']
3,Review and Feedback on Wireframes,24.000000,['John Doe']
4,Finalize Design,16.000000,['Bob Smith']
5,Development of Responsive Interface,80.000000,['Jane Doe']
6,Develop User-Friendly Navigation,32.000000,['Jane Doe']
7,Create 'About Us' Page,24.000000,"['Jane Doe', 'Bob Smith']"
8,Create 'Services' Page,32.000000,"['Jane Doe', 'Bob Smith']"
9,Create 'Contact Us' Page,24.000000,['Jane Doe']


In [20]:
#Inspecting milestone
milestomes = result.pydantic.dict()['milestones']
df_milestones = pd.DataFrame(milestomes)

#Display the dataframe as an HTML table
df_milestones.style.set_table_attributes('border="1"').set_caption("Task Details").set_table_styles(
    [{'selector': 'th, td', 'props': [{'font-size', '120%'}]}]
)

,milestone_name,tasks
0,Kickoff and Requirement Gathering,"['1', '2']"
1,Design Phase Completion,"['3', '4', '5']"
2,Development Phase Completion,"['6', '7', '8', '9', '10', '11', '12', '13', '14']"
3,Quality Assurance and Final Adjustments,"['15', '16']"
4,Website Launch,['17']
